# MalthusJAX quickstart

Run a native genetic algorithm with the base wheel. Use a Python 3.12 notebook kernel with Matplotlib installed for plotting.


In [ ]:
import malthusjax as mj
import matplotlib.pyplot as plt

print(f"✓ MalthusJAX Version: {mj.__version__}")

## Run a native genetic algorithm

Minimize the four-dimensional Sphere objective.


In [ ]:
import math

composer = mj.Composer.create_default()
result = composer.quick_run(
    fitness="native_sphere",
    genome_length=4,
    pop_size=16,
    generations=2,
    seeds=(42,),
    output_dir="notebook-results/first-run",
)
run = result.runs[0]
assert run.status == "success", run.error
assert math.isfinite(run.metrics["best_fitness"])
print(run.metrics["best_fitness"])


## 2️⃣ Inspecting Convergence & History

Every run produces rich telemetry, including generation-by-generation best and mean fitness values.

In [ ]:
# Access the first seed's run result
first_run = result.runs[0]
print(f"Seed {first_run.seed} Best Fitness: {first_run.metrics['best_fitness']:.4e}")

# Extract generations and fitness trajectory
generations = [h["generation"] for h in first_run.history]
best_fitness = [h["best_fitness"] for h in first_run.history]
mean_fitness = [h["mean_fitness"] for h in first_run.history]

# Plot convergence curve
plt.figure(figsize=(8, 4))
plt.plot(generations, best_fitness, color="#2563EB", lw=2, label=f"Best (Seed {first_run.seed})")
plt.plot(generations, mean_fitness, color="#93C5FD", lw=1.5, linestyle="--", label="Population Mean")
plt.xlabel("Generation", fontsize=11)
plt.ylabel("Fitness (Closer to 0 is better)", fontsize=11)
plt.title("MalthusJAX GA Convergence (Sphere 4D)", fontsize=13, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 3️⃣ Customizing Genetic Operators Declaratively

MalthusJAX features a built-in catalog of operators that can be configured with string parameters.
Here, we solve the **Sphere** function using **Tournament Selection**, **Blend Crossover**, and **Gaussian Mutation**:

In [ ]:
sphere_result = composer.quick_run(
    fitness="native_sphere",
    genome_length=10,
    selection="tournament:tournament_size=4",
    crossover="blend:alpha=0.5",
    mutation="gaussian:mutation_rate=0.15,mutation_strength=0.1",
    pop_size=100,
    generations=150,
    seeds=(42,),
)

r_summary = sphere_result.aggregated_summary()
print(f"✓ Sphere Best Fitness: {r_summary['best_fitness']['mean']:.4e}")

## 4️⃣ Benchmarking Algorithm Variants (`composer.compare`)

Want to empirically compare Tournament Selection vs Roulette Selection?
`composer.compare()` executes multiple algorithmic variants with **identical initial populations and seeds** for statistically rigorous comparison:

In [ ]:
comparison = composer.compare(
    pipelines={
        "Tournament": dict(selection="tournament:tournament_size=3"),
        "Roulette": dict(selection="roulette:num_selections=32"),
    },
    fitness="native_sphere",
    genome_length=10,
    pop_size=64,
    generations=50,
    seeds=(10, 20, 30),
)

for name, pipeline in comparison.pipelines.items():
    assert all(run.status == "success" for run in pipeline.runs), pipeline.runs
    best = pipeline.aggregated_summary()["best_fitness"]["mean"]
    print(f"{name}: mean best fitness {best:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
comparison.plot_convergence(seed_index=0, ax=ax)
ax.set_title("Tournament vs. roulette on seed 10")
plt.tight_layout()
plt.show()


---

## 🗺️ Next Steps: Choosing Your Abstraction Level

Depending on what you are building, pick your level:

| Goal | Recommended Path | Guide / Notebook |
| :--- | :--- | :--- |
| **Run sweeps & experiments** | **Level 4: Composer** | [`composer_technical_overview.ipynb`](composer_technical_overview.ipynb) / [`level_4_composer_guide.md`](../docs/framework_guides/level_4_composer_guide.md) |
| **Customize evolutionary loops** | **Level 3: Engines** | [`engine_technical_overview.ipynb`](engine_technical_overview.ipynb) / [`level_3_engines_guide.md`](../docs/framework_guides/level_3_engines_guide.md) |
| **Vectorized genetic operators** | **Level 2: Operators** | [`crossover_technical_overview.ipynb`](crossover_technical_overview.ipynb) / [`mutation_technical_overview.ipynb`](mutation_technical_overview.ipynb) |
| **Novel PyTrees / Composable Fitness** | **Level 1: Core** | [`genome_technical_overview.ipynb`](genome_technical_overview.ipynb) / [`level_1_core_guide.md`](../docs/framework_guides/level_1_core_guide.md) |
| **Scaffold new components** | **CLI Tool** | Run `make scaffold` or `python scripts/scaffold.py -h` |